### Converting Excel into csv with none spaced column names

In [1]:
import os
import pandas as pd

In [2]:
AGENCY_DIR = '../kingcounty_transit'

data = {
  "calendar": {
    "csv": "calendar.txt",
    "df": None
  },
  "routes": {
    "csv": "routes.txt",
    "df": None
  },
  "stops": {
    "csv": "stops.txt",
    "df": None
  },
  "stop_times": {
    "csv": "stop_times.txt",
    "df": None
  },
  "trips": {
    "csv": "trips.txt",
    "df": None
  },
}

for header, vals in data.items():
  data_csv =  os.path.join(AGENCY_DIR, vals['csv'])
  data_df = pd.read_csv(data_csv)
  data[header]['df'] = data_df

In [3]:
for header, vals in data.items():
  print(f"{header} has {len(vals['df'])} rows")
  print(vals['df'].head())
  print("")

calendar has 56 rows
   service_id  monday  tuesday  wednesday  thursday  friday  saturday  sunday  \
0      100658       1        1          1         1       1         0       0   
1        1048       0        0          0         0       1         0       0   
2       10857       0        0          0         0       0         0       0   
3      121408       0        1          1         1       0         0       0   
4      122175       0        1          1         1       1         0       0   

   start_date  end_date  
0    20260330  20260828  
1    20260227  20260227  
2    20260401  20260617  
3    20260224  20260828  
4    20260224  20260828  

routes has 144 rows
   route_id  agency_id route_short_name  route_long_name  \
0    100001          1                1              NaN   
1    100002          1               10              NaN   
2    100003          1              101              NaN   
3    100004          1              105              NaN   
4    100005    

In [7]:
import os
import json
import pandas as pd
from collections import defaultdict

AGENCY_DIR = '../kingcounty_transit'

data = {
    "calendar":   {"csv": "calendar.txt",   "df": None},
    "routes":     {"csv": "routes.txt",     "df": None},
    "stops":      {"csv": "stops.txt",      "df": None},
    "stop_times": {"csv": "stop_times.txt", "df": None},
    "trips":      {"csv": "trips.txt",      "df": None},
}

for header, vals in data.items():
    data_csv = os.path.join(AGENCY_DIR, vals['csv'])
    data[header]['df'] = pd.read_csv(data_csv)

stops_df      = data['stops']['df']
stop_times_df = data['stop_times']['df']
trips_df      = data['trips']['df']
calendar_df   = data['calendar']['df']
routes_df     = data['routes']['df']

DAY_COLS = ['monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday', 'sunday']

# Build list of day indices per service_id, e.g. [0, 1, 2, 3, 4]
calendar_df['day_indices'] = calendar_df[DAY_COLS].apply(
    lambda row: [str(i) for i, val in enumerate(row) if val == 1],
    axis=1
)

# Join stop_times -> trips -> calendar -> stops -> routes
merged = stop_times_df[['trip_id', 'stop_id', 'arrival_time']].merge(
    trips_df[['trip_id', 'route_id', 'service_id']],
    on='trip_id', how='left'
).merge(
    calendar_df[['service_id', 'day_indices']],
    on='service_id', how='left'
).merge(
    stops_df[['stop_id', 'stop_lat', 'stop_lon', 'stop_name']],
    on='stop_id', how='left'
).merge(
    routes_df[['route_id', 'route_short_name']],
    on='route_id', how='left'
)

# Use defaultdict to accumulate times per day
stops_json = {}

for _, row in merged.iterrows():
    stop_id  = str(int(row['stop_id']))
    route_id = str(int(row['route_id']))
    day_indices = row['day_indices'] if isinstance(row['day_indices'], list) else []

    if stop_id not in stops_json:
        stops_json[stop_id] = {
            "name":   row['stop_name'],
            "lat":    row['stop_lat'],
            "lon":    row['stop_lon'],
            "routes": {}
        }

    if route_id not in stops_json[stop_id]['routes']:
        stops_json[stop_id]['routes'][route_id] = {
            "short_name": str(row['route_short_name']),
            "trips": defaultdict(set)
        }

    for day in day_indices:
        stops_json[stop_id]['routes'][route_id]['trips'][day].add(row['arrival_time'])

# Convert sets to sorted lists and defaultdicts to plain dicts
for stop in stops_json.values():
    for route in stop['routes'].values():
        route['trips'] = {
            day: sorted(times)
            for day, times in sorted(route['trips'].items(), key=lambda x: int(x[0]))
        }

output_path = 'stops_locations.json'
with open(output_path, 'w') as f:
    json.dump(stops_json, f, separators=(',', ':'))

print(f"Done! Written to {output_path}")
print(f"Total stops: {len(stops_json)}")

Done! Written to stops_locations.json
Total stops: 6363


In [4]:
import os
import json
import pandas as pd

AGENCY_DIR = '../kingcounty_transit'

shapes_df = pd.read_csv(os.path.join(AGENCY_DIR, 'shapes.txt'))
trips_df  = pd.read_csv(os.path.join(AGENCY_DIR, 'trips.txt'))

# Build shape_id -> route_id mapping (one shape may be used by multiple trips,
# just take the first route_id we see for each shape)
shape_to_route = (
    trips_df[['shape_id', 'route_id']]
    .drop_duplicates('shape_id')
    .set_index('shape_id')['route_id']
    .to_dict()
)

shapes_json = {}

for shape_id, group in shapes_df.groupby('shape_id'):
    route_id = shape_to_route.get(shape_id)
    if route_id is None:
        continue
    pts = (
        group.sort_values('shape_pt_sequence')[['shape_pt_lat', 'shape_pt_lon']]
        .values.tolist()
    )
    # Downsample — every 6th point is plenty for a visual line
    # pts = pts[::6]
    route_key = str(int(route_id))
    # A route may have multiple shapes (e.g. inbound vs outbound directions)
    if route_key not in shapes_json:
        shapes_json[route_key] = []
    shapes_json[route_key].append(pts)

with open('route_shapes.json', 'w') as f:
    json.dump(shapes_json, f, separators=(',', ':'))

print(f"Done — {len(shapes_json)} routes written to route_shapes.json")

Done — 144 routes written to route_shapes.json


In [5]:
import os
import json
import pandas as pd

AGENCY_DIR = '../kingcounty_transit'
routes_df = pd.read_csv(os.path.join(AGENCY_DIR, 'routes.txt'))

colors = set()
route_meta = {}
for _, row in routes_df.iterrows():
    route_id = str(int(row['route_id']))
    route_meta[route_id] = {
        "short_name": str(row['route_short_name']),
        "desc": str(row['route_desc']) if pd.notna(row['route_desc']) else "",
        "color": str(row['route_color']) if pd.notna(row['route_color']) else "FFFFFF",
    }
    colors.add(str(row['route_color']) if pd.notna(row['route_color']) else "FFFFFF")

with open('route_meta.json', 'w') as f:
    json.dump(route_meta, f, separators=(',', ':'))

print(f"Done — {len(route_meta)} routes written to routes_meta.json")
print(colors)

Done — 144 routes written to routes_meta.json
{'32CD32', '2B376E', 'F47836', '9C182F', 'FFFFFF', 'FDB71A', '4298CC'}
